(document_grounding)=
# Document Grounding

The Document Grounding is a module in the [Orchestration Service](https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/orchestration-service.html).

The Document Grounding implements the Retrieval Augmented Generation (RAG) approach. It leverages the SAP Hana Vector Engine to retrieve info from relevant documents i.e., the "context" and uses them to generate more accurate responses.

## Prerequisites
A vector knowledge base is [required to use the Document Grounding module](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/grounding#prerequisites).

The vector knowledge base can be created from
  - a collection of documents in a [sharepoint folder, S3 storage, or an SFTP repository](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/grounding#option-1-upload-the-documents-to-the-supported-data-repository-and-run-data-pipeline), or
  - feeding text (chunks) directly [via Vector API](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/grounding#option-2-provide-the-chunks-of-document-via-vector-api-directly).

Another option is to use a website which provides elastic search capabilities.
At the moment, only the help.sap.com is supported.

(document_storage)=
### Create a Vector knowledge base
In this example, we will use an S3 data storage, which has been created by the user. The user has uploaded a set of documents to the S3 bucket.

Check if
 - [Document Grounding is enabled](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/create-resource-group-for-ai-data-management?q=document%20grounding) and
 - a [Generic Secret for the S3 bucket is created in AI Core](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/generic-secrets-for-grounding?q=document%20grounding), which allows for retrieving the documents in the S3 bucket.

The [Pipelines API](https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/pipeline-api-a9badce6a4da4df68e98549d64aa2217?q=document%20grounding) can be run via this SDK:


In [ ]:
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.document_grounding import PipelineAPIClient, S3PipelineCreateRequest, CommonConfiguration

aicore_client = get_proxy_client()
pipelines_api_client = PipelineAPIClient(aicore_client)
generic_secret_s3_bucket = "<*** generic secret name for the S3 bucket ***>"
s3_config = S3PipelineCreateRequest(configuration=CommonConfiguration(destination=generic_secret_s3_bucket))
response = pipelines_api_client.create_pipeline(s3_config)
print(f"Reference the Vector knowledge base using the pipeline ID: {response.pipelineId}")
# check the status of the vectorization pipeline until it is completed
print(pipelines_api_client.get_pipeline_status(response.pipelineId))

## Configuration of the Grounding Module
Provide the Orchestration Service URL and create a client for the Orchestration Service.

In [ ]:
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.document_grounding import (GroundingModule, GroundingType, DataRepositoryType,
                                                                GroundingFilterSearch, DocumentGrounding,DocumentGroundingFilter)
from gen_ai_hub.orchestration.models.llm import LLM
orchestration_service_url = "https://api.ai.<*** cluster-name ***>.aws.ml.hana.ondemand.com/v2/inference/deployments/<*** deployment_id ***>"
orchestration_service = OrchestrationService(api_url=orchestration_service_url)

llm = LLM(
    name="gpt-4o-mini",
    parameters={
        'temperature': 0.0,
    }
)

### Create the configuration
1) Define the prompts
2) Define the Grounding Module configuration

In [ ]:
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue

prompt = Template(messages=[
        SystemMessage("You are an expert on SAP Product features."),
        UserMessage("""Context: {{ ?grounding_response }}
                       Question: What are the features of {{ ?product }}
                    """),
    ])

#### Grounding configuration for searching SAP Help via elastic search

In [ ]:
filters = [DocumentGroundingFilter(id="SAPHelp", data_repository_type=DataRepositoryType.URL.value)]

grounding_config = GroundingModule(type=GroundingType.DOCUMENT_GROUNDING_SERVICE.value,
                                   config=DocumentGrounding(input_params=["product"],
                                                            output_param="grounding_response",
                                                            filters=filters
                                                            )
                                  )

config = OrchestrationConfig(template= prompt, llm=llm, grounding=grounding_config)

response = orchestration_service.run(config=config,
                                     template_values=[TemplateValue("product", "Generative AI Hub")])

print(response.orchestration_result.choices[0].message.content)

#### Grounding configuration for searching a custom data repository
Assume the documentation for custom product extension is vectorized and stored in the Vector knowledge base which we created earlier from the S3 bucket.

In [ ]:
filters = [DocumentGroundingFilter(id="<*** product extension docs id ***>",
                                   data_repositories=["<*** data repository (retrieval api) referencing the S3 pipeline id ***>"],
                                   search_config=GroundingFilterSearch(max_chunk_count=3),
                                   data_repository_type=DataRepositoryType.VECTOR.value
                                   )]

grounding_config = GroundingModule(
            type=GroundingType.DOCUMENT_GROUNDING_SERVICE.value,
            config=DocumentGrounding(input_params=["product"], output_param="grounding_response", filters=filters)
                   )

config = OrchestrationConfig(template=prompt, llm=llm, grounding=grounding_config)

response = orchestration_service.run(config=config,
                                     template_values=[TemplateValue("product", "<*** custom extension name ***>")])

print(response.orchestration_result.choices[0].message.content)

#### One can also show the retrieved context from the grounding module, which is added to the prompt for improving the response.

In [ ]:
print(response.module_results.grounding.data['grounding_result'])

(data-masking)=

#### Data Masking of the retrieved context
The retrieved context can be masked in the same way as in the Orchestration Service to avoid passing sensitive information to the LLM.

In [ ]:
from gen_ai_hub.orchestration.models.sap_data_privacy_integration import SAPDataPrivacyIntegration, MaskingMethod, ProfileEntity
from gen_ai_hub.orchestration.models.data_masking import DataMasking

data_masking = DataMasking(
    providers=[
        SAPDataPrivacyIntegration(
            method=MaskingMethod.ANONYMIZATION,
            entities=[ProfileEntity.SAP_IDS_INTERNAL],
            mask_grounding_input=True
        )
    ]
)
masking_config = OrchestrationConfig(template=prompt, llm=llm, grounding=grounding_config, data_masking=data_masking)
response = orchestration_service.run(config=masking_config,
                                     template_values=[TemplateValue("product", "<*** custom extension name ***>")])

print(response.orchestration_result.choices[0].message.content)

print(response.module_results.grounding.data['grounding_result'])